# EDA: Wait Times

Purpose: inspect loaded aggregate wait-time data from public Canadian sources, review missingness and descriptive statistics, and create a first leadership-facing chart for median wait by procedure at the Canada level.


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine


In [ ]:
host = os.getenv("POSTGRES_HOST", "localhost")
port = os.getenv("POSTGRES_PORT", "5432")
user = os.getenv("POSTGRES_USER", "healthops")
password = os.getenv("POSTGRES_PASSWORD", "healthops")
db = os.getenv("POSTGRES_DB", "healthops")
engine = create_engine(f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db}")


In [ ]:
query = """
SELECT
    f.source_name,
    p.procedure_name,
    g.geo_name,
    g.geo_type,
    g.province_code,
    f.reporting_year,
    f.reporting_period,
    f.median_wait_days,
    f.p90_wait_days,
    f.pct_meeting_benchmark,
    f.case_volume
FROM fact_wait_time AS f
JOIN dim_procedure AS p ON p.procedure_id = f.procedure_id
JOIN dim_geography AS g ON g.geo_id = f.geo_id
"""
df = pd.read_sql(query, engine)


In [ ]:
df.shape, df.dtypes, df.isna().mean().sort_values(ascending=False)


In [ ]:
df.describe(include="number")


In [ ]:
latest_year = df.loc[(df["source_name"] == "CIHI") & (df["geo_name"] == "Canada"), "reporting_year"].max()
plot_df = (
    df[(df["source_name"] == "CIHI") & (df["geo_name"] == "Canada") & (df["reporting_year"] == latest_year)]
    .dropna(subset=["median_wait_days"])
    .sort_values("median_wait_days", ascending=False)
)
ax = plot_df.plot.barh(x="procedure_name", y="median_wait_days", legend=False, figsize=(9, 7))
ax.set_title(f"Median wait by procedure, Canada, {int(latest_year)}")
ax.set_xlabel("Median wait (days)")
ax.set_ylabel("Procedure")
plt.tight_layout()


## What I See

- TODO: Fill in the strongest wait-time pattern after reviewing the chart.
- TODO: Fill in one data-quality or coverage caveat.
- TODO: Fill in one executive-reporting implication.
